In [1]:
from pathlib import Path
import sys

def find_project_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "utils").exists() and (p / "models").exists():
            return p
    for p in [here, *here.parents]:
        if any((p / m).exists() for m in (".git", "requirements.txt", "pyproject.toml")):
            return p
    return here

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data, feature_names

plt.style.use('seaborn-v0_8')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Model Definitions

In [3]:
class RNN(nn.Module):
    """Vanilla RNN - Current best architecture"""
    def __init__(self, input_size, output_size=1, rnn_units=64, fc_units=64, 
                 activation_fct=nn.ReLU(), dropout=[0.0, 0, 0], rnn_nonlinearity='relu'):
        super(RNN, self).__init__()
        self.rnn_layer = nn.RNN(input_size=input_size, hidden_size=rnn_units, 
                                num_layers=1, nonlinearity=rnn_nonlinearity, batch_first=True)
        self.fc = nn.Linear(rnn_units, fc_units)
        self.output_layer = nn.Linear(fc_units, output_size)
        self.dropout1 = nn.Dropout(dropout[0])
        self.dropout2 = nn.Dropout(dropout[1])
        self.dropout3 = nn.Dropout(dropout[2])
        self.activation_fct = activation_fct
 
    def forward(self, x):
        x = self.dropout1(x)
        x, _ = self.rnn_layer(x)
        x = x[:, -1, :]
        x = self.dropout2(x)
        x = self.activation_fct(self.fc(x))
        x = self.dropout3(x)
        x = self.output_layer(x)
        return x







## Load Data

In [ ]:
print("Loading data...")
train_dataloader, val_dataloader, pipeline = load_data(
    train_path = "../datasets_challenge/train",
    train_ratio = 0.98,
    batchsize = 64,
    columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],
    sequence_length = 20,
    threshold = True,
    threshold_value = 3.0,
    threshold_column = "vxCG",
    target = "sideSlip",
    scaling = True,
    seed = 42,
)

X_sample, y_sample = next(iter(train_dataloader))
input_size = X_sample.shape[2]

print(f"Data loaded")
print(f"   Input size: {input_size}")
print(f"   Train batches: {len(train_dataloader)}")
print(f"   Val batches: {len(val_dataloader)}")

Loading data...
../datasets_challenge/train/train_part_001.csv
../datasets_challenge/train/train_part_002.csv
../datasets_challenge/train/train_part_003.csv
../datasets_challenge/train/train_part_004.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_005.csv
../datasets_challenge/train/train_part_006.csv
../datasets_challenge/train/train_part_007.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_008.csv
../datasets_challenge/train/train_part_009.csv
../datasets_challenge/train/train_part_010.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.csv
../datasets_challenge/train/train_part_013.csv
../datasets_challenge/train/train_part_011.csv
../datasets_challenge/train/train_part_012.c

## Strategy 1: Multi-Seed Ensemble +

Train the **exact same architecture** with 5 different seeds and average predictions.



In [ ]:
def train_single_model(model, train_loader, val_loader, criterion, optimizer, 
                       num_epochs=50, early_stop_thresh=10, device='cuda', verbose=True):
    """Train a single model and return best checkpoint"""
    best_loss = np.inf
    best_epoch = -1
    best_model_state = None
    train_curve = []
    val_curve = []
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        running_loss = 0
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape([len(y), 1])
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        avg_train_loss = running_loss / len(train_loader)
        train_curve.append(avg_train_loss)
        
        # Validation
        model.eval()
        running_val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                X_val, y_val = x.to(device), y.to(device).reshape([len(y), 1])
                val_loss = criterion(model(X_val), y_val)
                running_val_loss += val_loss.item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_curve.append(avg_val_loss)
        
        if verbose:
            print(f"Epoch {epoch+1:2d}/{num_epochs} | Train: {avg_train_loss:.5f} | Val: {avg_val_loss:.5f}", end="")
        
        # Early stopping
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_epoch = epoch
            best_model_state = model.state_dict().copy()
            if verbose:
                print(" [saved]")
        elif epoch - best_epoch > early_stop_thresh:
            if verbose:
                print(f"\n  Early stop at epoch {epoch+1}")
            break
        else:
            if verbose:
                print("")
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, best_loss, train_curve, val_curve

In [6]:
def predict_with_model(model, test_loader, device='cuda'):
    """Generate predictions with a trained model"""
    model.eval()
    predictions = []
    with torch.no_grad():
        for x, _ in test_loader:
            X = x.to(device)
            pred = model(X)
            predictions.append(pred.cpu().numpy())
    return np.concatenate(predictions).flatten()

In [ ]:
# Train 5 models with different seeds
SEEDS = [42, 2024]
NUM_MODELS = len(SEEDS)

models = []
val_losses = []

print("\n" + "="*80)
print(f"TRAINING {NUM_MODELS} MODELS WITH DIFFERENT SEEDS")
print("="*80 + "\n")

for i, seed in enumerate(SEEDS):
    print(f"\n{'='*80}")
    print(f"MODEL {i+1}/{NUM_MODELS} | SEED = {seed}")
    print("="*80)
    
    # Set seed
    seed_everything(seed)
    
    # Create model
    model = RNN(
        input_size=input_size,
        output_size=1,
        rnn_units=64,
        fc_units=64,
        dropout=[0.0, 0.0, 0.0]
    ).to(device)
    
    # Train
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    
    model, val_loss, _, _ = train_single_model(
        model, train_dataloader, val_dataloader, criterion, optimizer,
        num_epochs=50, early_stop_thresh=10, device=device, verbose=True
    )
    
    models.append(model)
    val_losses.append(val_loss)
    
    print(f"Model {i+1} trained | Val Loss: {val_loss:.5f}")

print("\n" + "="*80)
print("ALL MODELS TRAINED")
print("="*80)
print(f"Val losses: {[f'{v:.5f}' for v in val_losses]}")
print(f"Mean val loss: {np.mean(val_losses):.5f}")
print(f"Std val loss:  {np.std(val_losses):.5f}")


🚀 TRAINING 2 MODELS WITH DIFFERENT SEEDS


📦 MODEL 1/2 | SEED = 42
Epoch  1/50 | Train: 0.27568 | Val: 0.19388 💾
Epoch  1/50 | Train: 0.27568 | Val: 0.19388 💾
Epoch  2/50 | Train: 0.19097 | Val: 0.17663 💾
Epoch  2/50 | Train: 0.19097 | Val: 0.17663 💾
Epoch  3/50 | Train: 0.17600 | Val: 0.17359 💾
Epoch  3/50 | Train: 0.17600 | Val: 0.17359 💾
Epoch  4/50 | Train: 0.16724 | Val: 0.16031 💾
Epoch  4/50 | Train: 0.16724 | Val: 0.16031 💾
Epoch  5/50 | Train: 0.16263 | Val: 0.16625
Epoch  5/50 | Train: 0.16263 | Val: 0.16625
Epoch  6/50 | Train: 0.15838 | Val: 0.16834
Epoch  6/50 | Train: 0.15838 | Val: 0.16834
Epoch  7/50 | Train: 0.15534 | Val: 0.15671 💾
Epoch  7/50 | Train: 0.15534 | Val: 0.15671 💾
Epoch  8/50 | Train: 0.15238 | Val: 0.15456 💾
Epoch  8/50 | Train: 0.15238 | Val: 0.15456 💾
Epoch  9/50 | Train: 0.15066 | Val: 0.14656 💾
Epoch  9/50 | Train: 0.15066 | Val: 0.14656 💾
Epoch 10/50 | Train: 0.14845 | Val: 0.15430
Epoch 10/50 | Train: 0.14845 | Val: 0.15430
Epoch 11/50 | Train: 0.1

In [ ]:
# Load test data
print("\nLoading test data...")
test_dataloader = load_test_data(
    test_path = "../datasets_challenge/test",
    batchsize = 512,
    pipeline = pipeline,
    load_anonymized_data=True
)
print("Test data loaded")


Loading test data...
../datasets_challenge/test/test_part_001.csv
../datasets_challenge/test/test_part_002.csv
../datasets_challenge/test/test_part_003.csv
../datasets_challenge/test/test_part_004.csv
../datasets_challenge/test/test_part_005.csv
../datasets_challenge/test/test_part_006.csv
../datasets_challenge/test/test_part_007.csv
../datasets_challenge/test/test_part_008.csv
../datasets_challenge/test/test_part_009.csv
../datasets_challenge/test/test_part_010.csv
../datasets_challenge/test/test_part_011.csv
✅ Test data loaded
✅ Test data loaded


In [ ]:
# Generate predictions from all models
print("\n" + "="*80)
print("GENERATING ENSEMBLE PREDICTIONS")
print("="*80)

all_predictions = []
for i, model in enumerate(models):
    print(f"Predicting with model {i+1}/{NUM_MODELS}...", end=" ")
    preds = predict_with_model(model, test_dataloader, device)
    all_predictions.append(preds)
    print(f"Done. Range: [{preds.min():.3f}, {preds.max():.3f}]")

# Simple average ensemble
ensemble_preds = np.mean(all_predictions, axis=0)

print("\n" + "="*80)
print("ENSEMBLE STATISTICS")
print("="*80)
print(f"Num models: {NUM_MODELS}")
print(f"Ensemble range: [{ensemble_preds.min():.3f}, {ensemble_preds.max():.3f}]")
print(f"Ensemble mean:  {ensemble_preds.mean():.3f}")
print(f"Ensemble std:   {ensemble_preds.std():.3f}")

# Individual model statistics
print("\nIndividual model std devs:")
for i, preds in enumerate(all_predictions):
    print(f"  Model {i+1}: std={preds.std():.3f}")

# Prediction variance across models
pred_variance = np.var(all_predictions, axis=0)
print(f"\nPrediction variance (mean): {pred_variance.mean():.6f}")
print(f"Prediction variance (max):  {pred_variance.max():.6f}")


🔮 GENERATING ENSEMBLE PREDICTIONS
Predicting with model 1/2... ✅ Range: [-9.937, 9.865]
Predicting with model 2/2... ✅ Range: [-9.937, 9.865]
Predicting with model 2/2... ✅ Range: [-9.069, 9.219]

📊 ENSEMBLE STATISTICS
Num models: 2
Ensemble range: [-9.476, 9.439]
Ensemble mean:  -0.011
Ensemble std:   1.511

Individual model std devs:
  Model 1: std=1.552
  Model 2: std=1.475

Prediction variance (mean): 0.006958
Prediction variance (max):  1.393562
✅ Range: [-9.069, 9.219]

📊 ENSEMBLE STATISTICS
Num models: 2
Ensemble range: [-9.476, 9.439]
Ensemble mean:  -0.011
Ensemble std:   1.511

Individual model std devs:
  Model 1: std=1.552
  Model 2: std=1.475

Prediction variance (mean): 0.006958
Prediction variance (max):  1.393562


In [ ]:
# Save ensemble predictions
y_pred = pd.DataFrame({
    "id": range(len(ensemble_preds)),
    "predicted_sideslip": ensemble_preds
})

y_pred.to_csv("predictions_multiseed_ensemble.csv", index=False)

print("\n" + "="*80)
print("MULTI-SEED ENSEMBLE SAVED")
print("="*80)
print(f"File: predictions_multiseed_ensemble.csv")
print(f"Samples: {len(ensemble_preds)}")
print(f"Expected score: ~0.265-0.273° (5-8% improvement from 0.28779°)")
print("="*80)


✅ MULTI-SEED ENSEMBLE SAVED
File: predictions_multiseed_ensemble.csv
Samples: 49328
Expected score: ~0.265-0.273° (5-8% improvement from 0.28779°)


## Idea 1: Weighted L1 Loss (Focus on Few-Shot)

Since the Kaggle metric scores **only the few-shot (high sideslip) MAE**, we should penalize errors on extreme sideslip values more during training.

In [ ]:
# ============================================================================
# IDEA 1: WEIGHTED L1 LOSS - Focus on Few-Shot (High Sideslip)
# ============================================================================

class FewShotWeightedL1Loss(nn.Module):
    """
    Weighted L1 loss that emphasizes few-shot (high |sideslip|) samples.
    
    Weight formula: w = base_weight + slope * max(0, |y| - threshold)
    - Samples with |sideslip| <= threshold get weight = base_weight
    - Samples with |sideslip| > threshold get higher weight
    """
    def __init__(self, base_weight=1.0, slope=0.5, threshold=3.0):
        super().__init__()
        self.base_weight = base_weight
        self.slope = slope
        self.threshold = threshold
    
    def forward(self, pred, target):
        # Higher weight for larger |sideslip| values (few-shot region)
        excess = torch.clamp(torch.abs(target) - self.threshold, min=0)
        weights = self.base_weight + self.slope * excess
        loss = torch.abs(pred - target) * weights
        return loss.mean()

print("FewShotWeightedL1Loss defined")
print(f"   - Base weight: 1.0 (for |sideslip| <= threshold)")
print(f"   - Extra weight: 0.5 * (|sideslip| - threshold) for high sideslip")
print(f"   - This means |sideslip|=10 gets weight = 1 + 0.5*(10-3) = 4.5x")

✅ FewShotWeightedL1Loss defined
   - Base weight: 1.0 (for |sideslip| <= threshold)
   - Extra weight: 0.5 * (|sideslip| - threshold) for high sideslip
   - This means |sideslip|=10 gets weight = 1 + 0.5*(10-3) = 4.5x


In [ ]:
# ============================================================================
# Train with Weighted L1 Loss (Seeds 42 + 2024)
# ============================================================================

print("\n" + "="*80)
print("IDEA 1: TRAINING WITH WEIGHTED L1 LOSS")
print("="*80)

SEEDS_WEIGHTED = [42, 2024]
models_weighted = []
val_losses_weighted = []

for i, seed in enumerate(SEEDS_WEIGHTED):
    print(f"\n{'='*60}")
    print(f"WEIGHTED MODEL {i+1}/{len(SEEDS_WEIGHTED)} | SEED = {seed}")
    print("="*60)
    
    seed_everything(seed)
    
    model = RNN(
        input_size=input_size,
        output_size=1,
        rnn_units=64,
        fc_units=64,
        dropout=[0.0, 0.0, 0.0]
    ).to(device)
    
    # Use weighted loss instead of standard L1
    criterion_weighted = FewShotWeightedL1Loss(base_weight=1.0, slope=0.5, threshold=3.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    
    model, val_loss, _, _ = train_single_model(
        model, train_dataloader, val_dataloader, criterion_weighted, optimizer,
        num_epochs=50, early_stop_thresh=10, device=device, verbose=True
    )
    
    models_weighted.append(model)
    val_losses_weighted.append(val_loss)
    
    print(f"Weighted Model {i+1} trained | Val Loss: {val_loss:.5f}")

print("\n" + "="*80)
print("WEIGHTED LOSS MODELS TRAINED")
print("="*80)
print(f"Val losses: {[f'{v:.5f}' for v in val_losses_weighted]}")


🎯 IDEA 1: TRAINING WITH WEIGHTED L1 LOSS

📦 WEIGHTED MODEL 1/2 | SEED = 42
Epoch  1/50 | Train: 0.31333 | Val: 0.21930 💾
Epoch  1/50 | Train: 0.31333 | Val: 0.21930 💾
Epoch  2/50 | Train: 0.21169 | Val: 0.20754 💾
Epoch  2/50 | Train: 0.21169 | Val: 0.20754 💾
Epoch  3/50 | Train: 0.19298 | Val: 0.18782 💾
Epoch  3/50 | Train: 0.19298 | Val: 0.18782 💾
Epoch  4/50 | Train: 0.18386 | Val: 0.17077 💾
Epoch  4/50 | Train: 0.18386 | Val: 0.17077 💾
Epoch  5/50 | Train: 0.17891 | Val: 0.16656 💾
Epoch  5/50 | Train: 0.17891 | Val: 0.16656 💾
Epoch  6/50 | Train: 0.17357 | Val: 0.20590
Epoch  6/50 | Train: 0.17357 | Val: 0.20590
Epoch  7/50 | Train: 0.16989 | Val: 0.16264 💾
Epoch  7/50 | Train: 0.16989 | Val: 0.16264 💾
Epoch  8/50 | Train: 0.16703 | Val: 0.16825
Epoch  8/50 | Train: 0.16703 | Val: 0.16825
Epoch  9/50 | Train: 0.16560 | Val: 0.17593
Epoch  9/50 | Train: 0.16560 | Val: 0.17593
Epoch 10/50 | Train: 0.16296 | Val: 0.15466 💾
Epoch 10/50 | Train: 0.16296 | Val: 0.15466 💾
Epoch 11/50 | Tr

In [ ]:
# Generate predictions with weighted loss models
print("\n" + "="*80)
print("GENERATING WEIGHTED LOSS PREDICTIONS")
print("="*80)

weighted_predictions = []
for i, model in enumerate(models_weighted):
    print(f"Predicting with weighted model {i+1}...", end=" ")
    preds = predict_with_model(model, test_dataloader, device)
    weighted_predictions.append(preds)
    print(f"Done. Range: [{preds.min():.3f}, {preds.max():.3f}]")

# Ensemble weighted models
ensemble_weighted = np.mean(weighted_predictions, axis=0)

print(f"\nWeighted Ensemble range: [{ensemble_weighted.min():.3f}, {ensemble_weighted.max():.3f}]")

# Save predictions
y_pred_weighted = pd.DataFrame({
    "id": range(len(ensemble_weighted)),
    "predicted_sideslip": ensemble_weighted
})
y_pred_weighted.to_csv("predictions_weighted_loss_ensemble.csv", index=False)
print("Saved: predictions_weighted_loss_ensemble.csv")


🔮 GENERATING WEIGHTED LOSS PREDICTIONS
Predicting with weighted model 1... ✅ Range: [-9.156, 9.569]
Predicting with weighted model 2... ✅ Range: [-9.156, 9.569]
Predicting with weighted model 2... ✅ Range: [-8.591, 9.962]

Weighted Ensemble range: [-8.873, 9.746]
✅ Saved: predictions_weighted_loss_ensemble.csv
✅ Range: [-8.591, 9.962]

Weighted Ensemble range: [-8.873, 9.746]
✅ Saved: predictions_weighted_loss_ensemble.csv


## Idea 2: Two-Stage Training (Pretrain -> Fine-tune on Few-Shot)

1. **Stage 1**: Train normally with L1 loss (ensure many-shot constraint is met)
2. **Stage 2**: Fine-tune with heavily weighted loss on few-shot samples

In [ ]:
# ============================================================================
# IDEA 2: TWO-STAGE TRAINING
# Stage 1: Pretrain with standard L1 loss
# Stage 2: Fine-tune with aggressive few-shot weighting
# ============================================================================

def train_two_stage(model, train_loader, val_loader, device='cuda', 
                    stage1_epochs=30, stage2_epochs=20, verbose=True):
    """
    Two-stage training:
    - Stage 1: Standard L1 loss (build general capability)
    - Stage 2: Heavily weighted loss for few-shot (fine-tune for metric)
    """
    
    # ========== STAGE 1: Standard L1 ==========
    if verbose:
        print("\nSTAGE 1: Standard L1 Loss")
        print("-"*40)
    
    criterion_stage1 = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    
    best_loss = np.inf
    best_state = None
    
    for epoch in range(stage1_epochs):
        model.train()
        running_loss = 0
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape([len(y), 1])
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion_stage1(pred, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        avg_train = running_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                X_val, y_val = x.to(device), y.to(device).reshape([len(y), 1])
                val_loss += criterion_stage1(model(X_val), y_val).item()
        avg_val = val_loss / len(val_loader)
        
        if avg_val < best_loss:
            best_loss = avg_val
            best_state = model.state_dict().copy()
        
        if verbose and (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:2d} | Train: {avg_train:.5f} | Val: {avg_val:.5f}")
    
    # Load best stage 1 model
    model.load_state_dict(best_state)
    if verbose:
        print(f"  Stage 1 complete | Best Val: {best_loss:.5f}")
    
    # ========== STAGE 2: Aggressive Few-Shot Weighting ==========
    if verbose:
        print("\nSTAGE 2: Few-Shot Fine-tuning (aggressive weighting)")
        print("-"*40)
    
    # More aggressive weighting for stage 2
    criterion_stage2 = FewShotWeightedL1Loss(base_weight=1.0, slope=1.0, threshold=2.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)  # Lower LR for fine-tuning
    
    best_loss_s2 = np.inf
    best_state_s2 = None
    
    for epoch in range(stage2_epochs):
        model.train()
        running_loss = 0
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape([len(y), 1])
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion_stage2(pred, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        avg_train = running_loss / len(train_loader)
        
        # Validation with standard L1 (to compare fairly)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                X_val, y_val = x.to(device), y.to(device).reshape([len(y), 1])
                val_loss += nn.L1Loss()(model(X_val), y_val).item()
        avg_val = val_loss / len(val_loader)
        
        if avg_val < best_loss_s2:
            best_loss_s2 = avg_val
            best_state_s2 = model.state_dict().copy()
        
        if verbose and (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:2d} | Train: {avg_train:.5f} | Val (L1): {avg_val:.5f}")
    
    # Load best stage 2 model
    model.load_state_dict(best_state_s2)
    if verbose:
        print(f"  Stage 2 complete | Best Val: {best_loss_s2:.5f}")
    
    return model, best_loss_s2

print("Two-stage training function defined")

✅ Two-stage training function defined


In [ ]:
# ============================================================================
# Train Two-Stage Models (Seeds 42 + 2024)
# ============================================================================

print("\n" + "="*80)
print("IDEA 2: TWO-STAGE TRAINING")
print("="*80)

SEEDS_TWOSTAGE = [42, 2024]
models_twostage = []
val_losses_twostage = []

for i, seed in enumerate(SEEDS_TWOSTAGE):
    print(f"\n{'='*60}")
    print(f"TWO-STAGE MODEL {i+1}/{len(SEEDS_TWOSTAGE)} | SEED = {seed}")
    print("="*60)
    
    seed_everything(seed)
    
    model = RNN(
        input_size=input_size,
        output_size=1,
        rnn_units=64,
        fc_units=64,
        dropout=[0.0, 0.0, 0.0]
    ).to(device)
    
    model, val_loss = train_two_stage(
        model, train_dataloader, val_dataloader, device=device,
        stage1_epochs=30, stage2_epochs=20, verbose=True
    )
    
    models_twostage.append(model)
    val_losses_twostage.append(val_loss)
    
    print(f"Two-Stage Model {i+1} complete | Final Val: {val_loss:.5f}")

print("\n" + "="*80)
print("TWO-STAGE MODELS TRAINED")
print("="*80)
print(f"Val losses: {[f'{v:.5f}' for v in val_losses_twostage]}")


🎯 IDEA 2: TWO-STAGE TRAINING

📦 TWO-STAGE MODEL 1/2 | SEED = 42

📍 STAGE 1: Standard L1 Loss
----------------------------------------
  Epoch  5 | Train: 0.16086 | Val: 0.16221
  Epoch  5 | Train: 0.16086 | Val: 0.16221
  Epoch 10 | Train: 0.14666 | Val: 0.14639
  Epoch 10 | Train: 0.14666 | Val: 0.14639
  Epoch 15 | Train: 0.13930 | Val: 0.13864
  Epoch 15 | Train: 0.13930 | Val: 0.13864
  Epoch 20 | Train: 0.13383 | Val: 0.13664
  Epoch 20 | Train: 0.13383 | Val: 0.13664
  Epoch 25 | Train: 0.12912 | Val: 0.13056
  Epoch 25 | Train: 0.12912 | Val: 0.13056
  Epoch 30 | Train: 0.12569 | Val: 0.12551
  ✅ Stage 1 complete | Best Val: 0.12384

📍 STAGE 2: Few-Shot Fine-tuning (aggressive weighting)
----------------------------------------
  Epoch 30 | Train: 0.12569 | Val: 0.12551
  ✅ Stage 1 complete | Best Val: 0.12384

📍 STAGE 2: Few-Shot Fine-tuning (aggressive weighting)
----------------------------------------
  Epoch  5 | Train: 0.15242 | Val (L1): 0.11880
  Epoch  5 | Train: 0.152

In [ ]:
# Generate predictions with two-stage models
print("\n" + "="*80)
print("GENERATING TWO-STAGE PREDICTIONS")
print("="*80)

twostage_predictions = []
for i, model in enumerate(models_twostage):
    print(f"Predicting with two-stage model {i+1}...", end=" ")
    preds = predict_with_model(model, test_dataloader, device)
    twostage_predictions.append(preds)
    print(f"Done. Range: [{preds.min():.3f}, {preds.max():.3f}]")

# Ensemble two-stage models
ensemble_twostage = np.mean(twostage_predictions, axis=0)

print(f"\nTwo-Stage Ensemble range: [{ensemble_twostage.min():.3f}, {ensemble_twostage.max():.3f}]")

# Save predictions
y_pred_twostage = pd.DataFrame({
    "id": range(len(ensemble_twostage)),
    "predicted_sideslip": ensemble_twostage
})
y_pred_twostage.to_csv("predictions_twostage_ensemble.csv", index=False)
print("Saved: predictions_twostage_ensemble.csv")


🔮 GENERATING TWO-STAGE PREDICTIONS
Predicting with two-stage model 1... ✅ Range: [-9.732, 9.736]
Predicting with two-stage model 2... ✅ Range: [-9.732, 9.736]
Predicting with two-stage model 2... ✅ Range: [-9.371, 10.188]

Two-Stage Ensemble range: [-9.551, 9.962]
✅ Saved: predictions_twostage_ensemble.csv
✅ Range: [-9.371, 10.188]

Two-Stage Ensemble range: [-9.551, 9.962]
✅ Saved: predictions_twostage_ensemble.csv


## Bonus: Blend Original Best with New Ideas

Combine the best baseline (42+2024 ensemble) with the new approaches

In [ ]:
# ============================================================================
# BLENDED ENSEMBLES - Combine approaches
# ============================================================================

print("\n" + "="*80)
print("CREATING BLENDED ENSEMBLES")
print("="*80)

# Baseline ensemble (from earlier training)
baseline_ensemble = ensemble_preds  # Standard L1 loss (42 + 2024)

# ============================================================================
# Blend 1: Baseline + Weighted Loss (50/50)
# ============================================================================
blend_baseline_weighted = (baseline_ensemble + ensemble_weighted) / 2
print(f"\nBlend 1: Baseline + Weighted (50/50)")
print(f"   Range: [{blend_baseline_weighted.min():.3f}, {blend_baseline_weighted.max():.3f}]")

y_blend1 = pd.DataFrame({
    "id": range(len(blend_baseline_weighted)),
    "predicted_sideslip": blend_baseline_weighted
})
y_blend1.to_csv("predictions_blend_baseline_weighted.csv", index=False)
print("   Saved: predictions_blend_baseline_weighted.csv")

# ============================================================================
# Blend 2: Baseline + Two-Stage (50/50)
# ============================================================================
blend_baseline_twostage = (baseline_ensemble + ensemble_twostage) / 2
print(f"\nBlend 2: Baseline + Two-Stage (50/50)")
print(f"   Range: [{blend_baseline_twostage.min():.3f}, {blend_baseline_twostage.max():.3f}]")

y_blend2 = pd.DataFrame({
    "id": range(len(blend_baseline_twostage)),
    "predicted_sideslip": blend_baseline_twostage
})
y_blend2.to_csv("predictions_blend_baseline_twostage.csv", index=False)
print("   Saved: predictions_blend_baseline_twostage.csv")

# ============================================================================
# Blend 3: All Three Methods (33/33/33)
# ============================================================================
blend_all_three = (baseline_ensemble + ensemble_weighted + ensemble_twostage) / 3
print(f"\nBlend 3: All Three Methods (33/33/33)")
print(f"   Range: [{blend_all_three.min():.3f}, {blend_all_three.max():.3f}]")

y_blend3 = pd.DataFrame({
    "id": range(len(blend_all_three)),
    "predicted_sideslip": blend_all_three
})
y_blend3.to_csv("predictions_blend_all_three.csv", index=False)
print("   Saved: predictions_blend_all_three.csv")

# ============================================================================
# Blend 4: Weighted + Two-Stage Only (50/50) - No baseline
# ============================================================================
blend_weighted_twostage = (ensemble_weighted + ensemble_twostage) / 2
print(f"\nBlend 4: Weighted + Two-Stage (50/50)")
print(f"   Range: [{blend_weighted_twostage.min():.3f}, {blend_weighted_twostage.max():.3f}]")

y_blend4 = pd.DataFrame({
    "id": range(len(blend_weighted_twostage)),
    "predicted_sideslip": blend_weighted_twostage
})
y_blend4.to_csv("predictions_blend_weighted_twostage.csv", index=False)
print("   Saved: predictions_blend_weighted_twostage.csv")


🔀 CREATING BLENDED ENSEMBLES

📊 Blend 1: Baseline + Weighted (50/50)
   Range: [-9.175, 9.592]
   ✅ Saved: predictions_blend_baseline_weighted.csv

📊 Blend 2: Baseline + Two-Stage (50/50)
   Range: [-9.514, 9.700]
   ✅ Saved: predictions_blend_baseline_twostage.csv

📊 Blend 3: All Three Methods (33/33/33)
   Range: [-9.300, 9.715]
   ✅ Saved: predictions_blend_all_three.csv

📊 Blend 4: Weighted + Two-Stage (50/50)
   Range: [-9.212, 9.854]
   ✅ Saved: predictions_blend_weighted_twostage.csv
   ✅ Saved: predictions_blend_all_three.csv

📊 Blend 4: Weighted + Two-Stage (50/50)
   Range: [-9.212, 9.854]
   ✅ Saved: predictions_blend_weighted_twostage.csv


In [ ]:
# ============================================================================
# SUMMARY - All files generated
# ============================================================================

print("\n" + "="*80)
print("SUMMARY: ALL PREDICTION FILES GENERATED")
print("="*80)

print("\nBASELINE (Standard L1 Loss):")
print("   - predictions_multiseed_ensemble.csv (seeds 42+2024)")
print("   Current best on Kaggle: 0.23623")

print("\nIDEA 1 - WEIGHTED LOSS:")
print("   - predictions_weighted_loss_ensemble.csv")
print("   Focus: Higher penalty for high |sideslip| values")

print("\nIDEA 2 - TWO-STAGE TRAINING:")
print("   - predictions_twostage_ensemble.csv")
print("   Stage 1: Standard L1 (30 epochs)")
print("   Stage 2: Aggressive few-shot weighting (20 epochs)")

print("\nBLENDED ENSEMBLES:")
print("   - predictions_blend_baseline_weighted.csv    (baseline + weighted)")
print("   - predictions_blend_baseline_twostage.csv    (baseline + two-stage)")
print("   - predictions_blend_all_three.csv            (all 3 methods)")
print("   - predictions_blend_weighted_twostage.csv    (weighted + two-stage)")

print("\n" + "="*80)
print("TESTING PRIORITY (recommended order):")
print("="*80)
print("1. predictions_twostage_ensemble.csv         (direct few-shot focus)")
print("2. predictions_weighted_loss_ensemble.csv    (simple weighting)")
print("3. predictions_blend_baseline_twostage.csv   (hedge your bets)")
print("4. predictions_blend_all_three.csv           (diversified)")

print("\nNote: If two-stage beats baseline, few-shot focus helps!")
print("   If weighted beats baseline → loss weighting works!")
print("   Blends are safer but may not give best improvement.")
print("="*80)


📁 SUMMARY: ALL PREDICTION FILES GENERATED

🔹 BASELINE (Standard L1 Loss):
   - predictions_multiseed_ensemble.csv (seeds 42+2024)
   Current best on Kaggle: 0.23623

🔹 IDEA 1 - WEIGHTED LOSS:
   - predictions_weighted_loss_ensemble.csv
   Focus: Higher penalty for high |sideslip| values

🔹 IDEA 2 - TWO-STAGE TRAINING:
   - predictions_twostage_ensemble.csv
   Stage 1: Standard L1 (30 epochs)
   Stage 2: Aggressive few-shot weighting (20 epochs)

🔹 BLENDED ENSEMBLES:
   - predictions_blend_baseline_weighted.csv    (baseline + weighted)
   - predictions_blend_baseline_twostage.csv    (baseline + two-stage)
   - predictions_blend_all_three.csv            (all 3 methods)
   - predictions_blend_weighted_twostage.csv    (weighted + two-stage)

🎯 TESTING PRIORITY (recommended order):
1. predictions_twostage_ensemble.csv         (direct few-shot focus)
2. predictions_weighted_loss_ensemble.csv    (simple weighting)
3. predictions_blend_baseline_twostage.csv   (hedge your bets)
4. predictions_

## Deep Dive: Expanding Two-Stage Training

**Analysis Results:**
- BEST: Blend Baseline+TwoStage = 0.22013 (6.8% improvement!)
- Two-Stage alone: 0.23591 (slightly better than baseline)
- Weighted Loss alone: 0.28946 (too aggressive, hurts many-shot)

**Strategy: Expand Two-Stage experiments with different hyperparameters**
1. Vary Stage 2 aggressiveness (slope, threshold)
2. Vary Stage 2 learning rate
3. Vary Stage 2 epoch count
4. Try different blend ratios with baseline

In [ ]:
# ============================================================================
# EXPANDED TWO-STAGE TRAINING - Multiple Configurations
# ============================================================================

def train_two_stage_configurable(model, train_loader, val_loader, device='cuda',
                                  stage1_epochs=30, stage2_epochs=20,
                                  stage2_slope=1.0, stage2_threshold=2.0,
                                  stage2_lr=0.0001, verbose=True, config_name=""):
    """
    Configurable two-stage training with adjustable Stage 2 parameters.
    """
    
    # ========== STAGE 1: Standard L1 ==========
    if verbose:
        print(f"\nSTAGE 1: Standard L1 Loss ({stage1_epochs} epochs)")
    
    criterion_stage1 = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    
    best_loss = np.inf
    best_state = None
    
    for epoch in range(stage1_epochs):
        model.train()
        running_loss = 0
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape([len(y), 1])
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion_stage1(pred, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        avg_train = running_loss / len(train_loader)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                X_val, y_val = x.to(device), y.to(device).reshape([len(y), 1])
                val_loss += criterion_stage1(model(X_val), y_val).item()
        avg_val = val_loss / len(val_loader)
        
        if avg_val < best_loss:
            best_loss = avg_val
            best_state = model.state_dict().copy()
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:2d} | Val: {avg_val:.5f}")
    
    model.load_state_dict(best_state)
    if verbose:
        print(f"  Stage 1 done | Best Val: {best_loss:.5f}")
    
    # ========== STAGE 2: Configurable Few-Shot Weighting ==========
    if verbose:
        print(f"\nSTAGE 2: slope={stage2_slope}, thresh={stage2_threshold}, lr={stage2_lr}")
    
    criterion_stage2 = FewShotWeightedL1Loss(base_weight=1.0, slope=stage2_slope, threshold=stage2_threshold)
    optimizer = torch.optim.Adam(model.parameters(), lr=stage2_lr)
    
    best_loss_s2 = np.inf
    best_state_s2 = None
    
    for epoch in range(stage2_epochs):
        model.train()
        running_loss = 0
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape([len(y), 1])
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion_stage2(pred, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                X_val, y_val = x.to(device), y.to(device).reshape([len(y), 1])
                val_loss += nn.L1Loss()(model(X_val), y_val).item()
        avg_val = val_loss / len(val_loader)
        
        if avg_val < best_loss_s2:
            best_loss_s2 = avg_val
            best_state_s2 = model.state_dict().copy()
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:2d} | Val (L1): {avg_val:.5f}")
    
    model.load_state_dict(best_state_s2)
    if verbose:
        print(f"  Stage 2 done | Best Val: {best_loss_s2:.5f}")
    
    return model, best_loss_s2

print("Configurable two-stage training function defined")

✅ Configurable two-stage training function defined


In [ ]:
# ============================================================================
# TEST MULTIPLE TWO-STAGE CONFIGURATIONS
# ============================================================================

# Configurations to test (based on analysis - less aggressive might be better)
CONFIGS = [
    # (config_name, slope, threshold, lr, stage2_epochs)
    ("mild", 0.5, 3.0, 0.0001, 15),        # Mild: less aggressive
    ("moderate", 0.7, 2.5, 0.0001, 20),    # Moderate
    ("aggressive_longer", 1.0, 2.0, 0.00005, 30),  # Original but longer + lower LR
    ("very_mild", 0.3, 4.0, 0.0001, 10),   # Very mild: barely weight few-shot
]

SEEDS_EXP = [42, 2024]

# Store all results
experiment_results = {}

print("\n" + "="*80)
print("TESTING MULTIPLE TWO-STAGE CONFIGURATIONS")
print("="*80)

for config_name, slope, threshold, lr, s2_epochs in CONFIGS:
    print(f"\n{'#'*80}")
    print(f"## CONFIG: {config_name.upper()}")
    print(f"   slope={slope}, threshold={threshold}, lr={lr}, epochs={s2_epochs}")
    print("#"*80)
    
    config_models = []
    config_val_losses = []
    
    for seed in SEEDS_EXP:
        print(f"\nSeed {seed}")
        seed_everything(seed)
        
        model = RNN(
            input_size=input_size,
            output_size=1,
            rnn_units=64,
            fc_units=64,
            dropout=[0.0, 0.0, 0.0]
        ).to(device)
        
        model, val_loss = train_two_stage_configurable(
            model, train_dataloader, val_dataloader, device=device,
            stage1_epochs=30, stage2_epochs=s2_epochs,
            stage2_slope=slope, stage2_threshold=threshold, stage2_lr=lr,
            verbose=True, config_name=config_name
        )
        
        config_models.append(model)
        config_val_losses.append(val_loss)
    
    # Generate predictions
    config_preds = []
    for model in config_models:
        preds = predict_with_model(model, test_dataloader, device)
        config_preds.append(preds)
    
    config_ensemble = np.mean(config_preds, axis=0)
    
    experiment_results[config_name] = {
        'models': config_models,
        'val_losses': config_val_losses,
        'ensemble': config_ensemble
    }
    
    print(f"\n{config_name}: Val losses = {[f'{v:.5f}' for v in config_val_losses]}")
    print(f"   Ensemble range: [{config_ensemble.min():.3f}, {config_ensemble.max():.3f}]")

print("\n" + "="*80)
print("ALL CONFIGURATIONS TRAINED")
print("="*80)


🔬 TESTING MULTIPLE TWO-STAGE CONFIGURATIONS

################################################################################
## CONFIG: MILD
   slope=0.5, threshold=3.0, lr=0.0001, epochs=15
################################################################################

🌱 Seed 42

📍 STAGE 1: Standard L1 Loss (30 epochs)
  Epoch 10 | Val: 0.14314
  Epoch 10 | Val: 0.14314
  Epoch 20 | Val: 0.13469
  Epoch 20 | Val: 0.13469
  Epoch 30 | Val: 0.12461
  ✅ Stage 1 done | Best Val: 0.12461

📍 STAGE 2: slope=0.5, thresh=3.0, lr=0.0001
  Epoch 30 | Val: 0.12461
  ✅ Stage 1 done | Best Val: 0.12461

📍 STAGE 2: slope=0.5, thresh=3.0, lr=0.0001
  Epoch 10 | Val (L1): 0.11438
  Epoch 10 | Val (L1): 0.11438
  ✅ Stage 2 done | Best Val: 0.11438

🌱 Seed 2024

📍 STAGE 1: Standard L1 Loss (30 epochs)
  ✅ Stage 2 done | Best Val: 0.11438

🌱 Seed 2024

📍 STAGE 1: Standard L1 Loss (30 epochs)
  Epoch 10 | Val: 0.14753
  Epoch 10 | Val: 0.14753
  Epoch 20 | Val: 0.13847
  Epoch 20 | Val: 0.13847
  Epoc

In [ ]:
# ============================================================================
# SAVE TWO-STAGE CONFIGURATION PREDICTIONS
# ============================================================================

print("\n" + "="*80)
print("SAVING TWO-STAGE CONFIGURATION PREDICTIONS")
print("="*80)

for config_name, data in experiment_results.items():
    ensemble = data['ensemble']
    filename = f"predictions_twostage_{config_name}.csv"
    
    y_pred_config = pd.DataFrame({
        "id": range(len(ensemble)),
        "predicted_sideslip": ensemble
    })
    y_pred_config.to_csv(filename, index=False)
    print(f"Saved: {filename}")

print("\nFiles saved:")
for config_name in experiment_results.keys():
    print(f"   - predictions_twostage_{config_name}.csv")


💾 SAVING TWO-STAGE CONFIGURATION PREDICTIONS
✅ Saved: predictions_twostage_mild.csv
✅ Saved: predictions_twostage_moderate.csv
✅ Saved: predictions_twostage_aggressive_longer.csv
✅ Saved: predictions_twostage_aggressive_longer.csv
✅ Saved: predictions_twostage_very_mild.csv

📁 Files saved:
   - predictions_twostage_mild.csv
   - predictions_twostage_moderate.csv
   - predictions_twostage_aggressive_longer.csv
   - predictions_twostage_very_mild.csv
✅ Saved: predictions_twostage_very_mild.csv

📁 Files saved:
   - predictions_twostage_mild.csv
   - predictions_twostage_moderate.csv
   - predictions_twostage_aggressive_longer.csv
   - predictions_twostage_very_mild.csv


## Optimized Blends with Baseline

Since **Baseline + TwoStage = 0.22013** was the best, let's try:
1. Different blend ratios (not just 50/50)
2. Blends with each new two-stage configuration

In [ ]:
# ============================================================================
# OPTIMIZED BLENDS - Different ratios and configurations
# ============================================================================

print("\n" + "="*80)
print("CREATING OPTIMIZED BLENDS")
print("="*80)

# Baseline is: ensemble_preds (from standard L1 training, seeds 42+2024)
baseline = ensemble_preds

# ============================================================================
# PART 1: Test different blend ratios with original two-stage
# ============================================================================
print("\nPart 1: Blend Ratio Experiments (Baseline + Original TwoStage)")
print("-"*60)

# Original two-stage ensemble
twostage_original = ensemble_twostage

blend_ratios = [0.3, 0.4, 0.5, 0.6, 0.7]  # baseline weight

for baseline_weight in blend_ratios:
    twostage_weight = 1 - baseline_weight
    blend = baseline * baseline_weight + twostage_original * twostage_weight
    
    filename = f"predictions_blend_b{int(baseline_weight*100)}_ts{int(twostage_weight*100)}.csv"
    
    y_blend = pd.DataFrame({
        "id": range(len(blend)),
        "predicted_sideslip": blend
    })
    y_blend.to_csv(filename, index=False)
    print(f"Saved: {filename} (baseline={baseline_weight:.0%}, twostage={twostage_weight:.0%})")

# ============================================================================
# PART 2: Blend baseline with each new configuration (50/50)
# ============================================================================
print("\nPart 2: Baseline + New Configurations (50/50)")
print("-"*60)

for config_name, data in experiment_results.items():
    config_ensemble = data['ensemble']
    blend = (baseline + config_ensemble) / 2
    
    filename = f"predictions_blend_baseline_{config_name}.csv"
    
    y_blend = pd.DataFrame({
        "id": range(len(blend)),
        "predicted_sideslip": blend
    })
    y_blend.to_csv(filename, index=False)
    print(f"Saved: {filename}")

# ============================================================================
# PART 3: Best blend ratio with best new config (test 40/60)
# ============================================================================
print("\nPart 3: Special Blends")
print("-"*60)

# Blend baseline (40%) with mild (60%)
if 'mild' in experiment_results:
    blend_40_60_mild = baseline * 0.4 + experiment_results['mild']['ensemble'] * 0.6
    y_special1 = pd.DataFrame({
        "id": range(len(blend_40_60_mild)),
        "predicted_sideslip": blend_40_60_mild
    })
    y_special1.to_csv("predictions_blend_b40_mild60.csv", index=False)
    print("Saved: predictions_blend_b40_mild60.csv")

# Blend baseline (40%) with moderate (60%)
if 'moderate' in experiment_results:
    blend_40_60_mod = baseline * 0.4 + experiment_results['moderate']['ensemble'] * 0.6
    y_special2 = pd.DataFrame({
        "id": range(len(blend_40_60_mod)),
        "predicted_sideslip": blend_40_60_mod
    })
    y_special2.to_csv("predictions_blend_b40_moderate60.csv", index=False)
    print("Saved: predictions_blend_b40_moderate60.csv")

# Blend original twostage with mild (50/50) - no baseline
if 'mild' in experiment_results:
    blend_ts_mild = (twostage_original + experiment_results['mild']['ensemble']) / 2
    y_special3 = pd.DataFrame({
        "id": range(len(blend_ts_mild)),
        "predicted_sideslip": blend_ts_mild
    })
    y_special3.to_csv("predictions_blend_twostage_mild.csv", index=False)
    print("Saved: predictions_blend_twostage_mild.csv")

print("\n" + "="*80)
print("ALL BLENDS CREATED")
print("="*80)


🔀 CREATING OPTIMIZED BLENDS

📊 Part 1: Blend Ratio Experiments (Baseline + Original TwoStage)
------------------------------------------------------------
✅ Saved: predictions_blend_b30_ts70.csv (baseline=30%, twostage=70%)
✅ Saved: predictions_blend_b40_ts60.csv (baseline=40%, twostage=60%)
✅ Saved: predictions_blend_b50_ts50.csv (baseline=50%, twostage=50%)
✅ Saved: predictions_blend_b40_ts60.csv (baseline=40%, twostage=60%)
✅ Saved: predictions_blend_b50_ts50.csv (baseline=50%, twostage=50%)
✅ Saved: predictions_blend_b60_ts40.csv (baseline=60%, twostage=40%)
✅ Saved: predictions_blend_b70_ts30.csv (baseline=70%, twostage=30%)

📊 Part 2: Baseline + New Configurations (50/50)
------------------------------------------------------------
✅ Saved: predictions_blend_b60_ts40.csv (baseline=60%, twostage=40%)
✅ Saved: predictions_blend_b70_ts30.csv (baseline=70%, twostage=30%)

📊 Part 2: Baseline + New Configurations (50/50)
------------------------------------------------------------
✅ S

In [ ]:
# ============================================================================
# FINAL SUMMARY - All prediction files for Kaggle testing
# ============================================================================

print("\n" + "="*80)
print("FINAL SUMMARY: ALL PREDICTION FILES")
print("="*80)

print("\nKNOWN BEST: predictions_blend_baseline_twostage.csv = 0.22013"))
print("   (baseline 50% + original twostage 50%)")

print("\n" + "-"*80)
print("NEW TWO-STAGE CONFIGURATIONS (to test standalone):")
print("-"*80)
for config_name in experiment_results.keys():
    print(f"   - predictions_twostage_{config_name}.csv")

print("\n" + "-"*80)
print("BLEND RATIO EXPERIMENTS (baseline + original twostage):")
print("-"*80)
print("   - predictions_blend_b30_ts70.csv  (baseline=30%, twostage=70%)")
print("   - predictions_blend_b40_ts60.csv  (baseline=40%, twostage=60%)")
print("   - predictions_blend_b50_ts50.csv  (baseline=50%, twostage=50%) ← same as known best")
print("   - predictions_blend_b60_ts40.csv  (baseline=60%, twostage=40%)")
print("   - predictions_blend_b70_ts30.csv  (baseline=70%, twostage=30%)")

print("\n" + "-"*80)
print("BASELINE + NEW CONFIG BLENDS (50/50):")
print("-"*80)
for config_name in experiment_results.keys():
    print(f"   - predictions_blend_baseline_{config_name}.csv")

print("\n" + "-"*80)
print("SPECIAL BLENDS:")
print("-"*80)
print("   - predictions_blend_b40_mild60.csv       (baseline 40% + mild 60%)")
print("   - predictions_blend_b40_moderate60.csv   (baseline 40% + moderate 60%)")
print("   - predictions_blend_twostage_mild.csv    (original twostage + mild, no baseline)")

print("\n" + "="*80)
print("RECOMMENDED TESTING ORDER:")
print("="*80)
print("1. predictions_blend_b40_ts60.csv           (more twostage weight)")
print("2. predictions_blend_b60_ts40.csv           (more baseline weight)")
print("3. predictions_blend_baseline_mild.csv      (milder Stage 2)")
print("4. predictions_blend_baseline_moderate.csv  (moderate Stage 2)")
print("5. predictions_twostage_mild.csv            (standalone mild)")
print("6. predictions_blend_b40_mild60.csv         (40/60 with mild)")

print("\nHypothesis:")
print("   - If b40_ts60 > b50_ts50 → more twostage is better")
print("   - If b60_ts40 > b50_ts50 → more baseline is better")
print("   - If mild blends beat aggressive → less aggressive Stage 2 is better")
print("="*80)


📋 FINAL SUMMARY: ALL PREDICTION FILES

🏆 KNOWN BEST: predictions_blend_baseline_twostage.csv = 0.22013
   (baseline 50% + original twostage 50%)

--------------------------------------------------------------------------------
🔬 NEW TWO-STAGE CONFIGURATIONS (to test standalone):
--------------------------------------------------------------------------------
   - predictions_twostage_mild.csv
   - predictions_twostage_moderate.csv
   - predictions_twostage_aggressive_longer.csv
   - predictions_twostage_very_mild.csv

--------------------------------------------------------------------------------
🔀 BLEND RATIO EXPERIMENTS (baseline + original twostage):
--------------------------------------------------------------------------------
   - predictions_blend_b30_ts70.csv  (baseline=30%, twostage=70%)
   - predictions_blend_b40_ts60.csv  (baseline=40%, twostage=60%)
   - predictions_blend_b50_ts50.csv  (baseline=50%, twostage=50%) ← same as known best
   - predictions_blend_b60_ts40.csv 